# Sweeping a parameter

One scenario per value of the unsecured arm's `premium`, over 200,000 applications in batches of 50,000.
The `Progress` callback reports after every batch, and no metric holds the whole input.

**How an experiment runs.** The runner feeds the *baseline* (the unchanged flow) and every *scenario* the
same records, one batch at a time. Each **metric** gets its own copy per scenario and collects through hooks:
a *global* metric reads the flow's output; a *step* metric (`nodes="<glob>"`) reads what matching steps
wrote, on the records that ran them. `summarise()` gives a typed summary; a **plot** draws one metric's
summaries across scenarios. `result.show()` lays it all out.

In [1]:
from decider.experiments import *
from example_projects.experiments.loan_pipeline import applications, build

ENTRY = "example_projects.experiments.loan_pipeline:build"
pipeline = build()   # ratio -> rate -> pricing (secured | unsecured) -> approved, payment
loans = applications(200_000)
loans.head()

income,debt,collateral
f64,f64,f64
3829.0,718.0,1344.0
3454.0,938.0,3913.0
4704.0,1956.0,0.0
3797.0,1960.0,16821.0
2939.0,387.0,0.0


In [2]:
exp = Experiment("premium-sweep", pipeline, loans, entry=ENTRY, batch_size=50_000)
for premium in (0.02, 0.06, 0.08):
    exp.scenario(f"premium_{premium}", params={"loans": {"pricing": {"unsecured": {"premium": premium}}}})
exp.metric(
    Divergence(column="approved"),                                    # global
    ValueCounts(column="approved"),                                   # global
    Histogram(name="payment", column="payment", range=(0, 400)),      # global
    Histogram(name="unsecured_rate", column="rate", nodes="loans/pricing/unsecured", range=(0.1, 0.35)),
)
exp.plot(Changes(metric="divergence"), Distribution(metric="unsecured_rate"),
         Distribution(name="payments", metric="payment"))
exp.callback(Progress())
result = exp.run()

premium-sweep: 50,000/200,000 records, 0.4s
premium-sweep: 100,000/200,000 records, 0.4s
premium-sweep: 150,000/200,000 records, 0.5s
premium-sweep: 200,000/200,000 records, 0.5s


The unsecured arm's rate, binned the same way for every scenario; click a scenario in the legend to pick
it out. The pipeline's own premium is 0.04, so the baseline is the reference.

In [3]:
result.plots["unsecured_rate"].show()

Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'baseline', 'start': 0.1125, 'end': 0.125, 'records': 0}, {'scenario': 'baseline', 'start': 0.125, 'end': 0.1375, 'records': 0}, {'scenario': 'baseline', 'start': 0.1375, 'end': 0.15, 'records': 10838}, {'scenario': 'baseline', 'start': 0.15, 'end': 0.1625, 'records': 26335}, {'scenario': 'baseline', 'start': 0.1625, 'end': 0.175, 'records': 22676}, {'scenario': 'baseline', 'start': 0.175, 'end': 0.1875, 'records': 13279}, {'scenario': 'baseline', 'start': 0.1875, 'end': 0.2, 'records': 5427}, {'scenario': 'baseline', 'start': 0.2, 'end': 0.2125, 'records': 1309}, {'scenario': 'baseline', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 133}, {'scenario': 'baseline', 'start': 0.22499999999999998, 'end': 0.2375, 'records': 0}, {'scenario': 'baseline', 'start': 0.2375, 'end': 0.25, 'records': 0}, {'scenario': 'baseline', 'start': 0.25, 'end': 0.26249999999999996, 'records': 0}, {'scenario': 'baseline', 'start': 0.26249999999999996, 'end': 0.275, 'records': 0}, {'scenario': 'baseline', 'start': 0.275, 'end': 0.2875, 'records': 0}, {'scenario': 'baseline', 'start': 0.2875, 'end': 0.3, 'records': 0}, {'scenario': 'baseline', 'start': 0.3, 'end': 0.3125, 'records': 0}, {'scenario': 'baseline', 'start': 0.3125, 'end': 0.32499999999999996, 'records': 0}, {'scenario': 'baseline', 'start': 0.32499999999999996, 'end': 0.3375, 'records': 0}, {'scenario': 'baseline', 'start': 0.3375, 'end': 0.35, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.1125, 'end': 0.125, 'records': 4949}, {'scenario': 'premium_0.02', 'start': 0.125, 'end': 0.1375, 'records': 32224}, {'scenario': 'premium_0.02', 'start': 0.1375, 'end': 0.15, 'records': 27513}, {'scenario': 'premium_0.02', 'start': 0.15, 'end': 0.1625, 'records': 12220}, {'scenario': 'premium_0.02', 'start': 0.1625, 'end': 0.175, 'records': 2903}, {'scenario': 'premium_0.02', 'start': 0.175, 'end': 0.1875, 'records': 188}, {'scenario': 'premium_0.02', 'start': 0.1875, 'end': 0.2, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2, 'end': 0.2125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.22499999999999998, 'end': 0.2375, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2375, 'end': 0.25, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.25, 'end': 0.26249999999999996, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.26249999999999996, 'end': 0.275, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.275, 'end': 0.2875, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2875, 'end': 0.3, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3, 'end': 0.3125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3125, 'end': 0.32499999999999996, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.32499999999999996, 'end': 0.3375, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3375, 'end': 0.35, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1125, 'end': 0.125, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.125, 'end': 0.1375, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1375, 'end': 0.15, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.15, 'end': 0.1625, 'records': 587}, {'scenario': 'premium_0.06', 'start': 0.1625, 'end': 0.175, 'records': 14629}, {'scenario': 'premium_0.06', 'start': 0.175, 'end': 0.1875, 'records': 21957}, {'scenario': 'premium_0.06', 'start': 0.1875, 'end': 0.2, 'records': 19133}, {'scenario': 'premium_0.06', 'start': 0.2, 'end': 0.2125, 'records': 13162}, {'scenario': 'premium_0.06', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 6874}, 

In [4]:
result.show()

### premium-sweep: completed

200,000 records · revision `64edd7e966f0` · 3 scenario(s) next to the baseline

scenario,status,records,revision,changes,error
str,str,i64,null,str,null
"""baseline""","""completed""",200000,null,"""unchanged""",null
"""premium_0.02""","""completed""",200000,null,"""params {""loans"": {""pricing"": {…",null
"""premium_0.06""","""completed""",200000,null,"""params {""loans"": {""pricing"": {…",null
"""premium_0.08""","""completed""",200000,null,"""params {""loans"": {""pricing"": {…",null


#### divergence

`divergence` of `approved` in the flow's output

scenario,column,compared,changed,first,samples
str,str,i64,i64,i64,str
"""baseline""","""approved""",200000,0,null,"""[]"""
"""premium_0.02""","""approved""",200000,0,null,"""[]"""
"""premium_0.06""","""approved""",200000,4986,22,"""[{""row"": 22, ""expected"": true,…"
"""premium_0.08""","""approved""",200000,42720,14,"""[{""row"": 14, ""expected"": true,…"


Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'changed': 0, 'compared': 200000, 'first': None}, {'scenario': 'premium_0.02', 'changed': 0, 'compared': 200000, 'first': None}, {'scenario': 'premium_0.06', 'changed': 4986, 'compared': 200000, 'first': 22}, {'scenario': 'premium_0.08', 'changed': 42720, 'compared': 200000, 'first': 14}]}, 'mark': {'type': 'bar'}, 'encoding': {'tooltip': [{'field': 'scenario', 'type': 'nominal'}, {'field': 'changed', 'type': 'quantitative'}, {'field': 'compared', 'type': 'quantitative'}, {'field': 'first', 'title': 'first changed record', 'type': 'quantitative'}], 'x': {'field': 'scenario', 'sort': None, 'type': 'nominal'}, 'y': {'field': 'changed', 'title': 'records changed', 'type': 'quantitative'}}, 'title': 'approved changed vs baseline', '$schema': 'https://vega.github.io/schema/vega-lite/v6.4.1.json'})

...

scenario,column,edges,counts,below,above,nulls
str,str,str,str,i64,i64,i64
"""baseline""","""payment""","""[0, 20, 40, 60, 80, 100, …] (2…","""[11498, 24604, 28183, 26679, 2…",0,1463,0
"""premium_0.02""","""payment""","""[0, 20, 40, 60, 80, 100, …] (2…","""[11646, 24942, 28372, 26869, 2…",0,1358,0
"""premium_0.06""","""payment""","""[0, 20, 40, 60, 80, 100, …] (2…","""[11365, 24311, 27909, 26569, 2…",0,1563,0
"""premium_0.08""","""payment""","""[0, 20, 40, 60, 80, 100, …] (2…","""[11251, 24025, 27733, 26298, 2…",0,1680,0


Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.0, 'end': 20.0, 'records': 11498}, {'scenario': 'baseline', 'start': 20.0, 'end': 40.0, 'records': 24604}, {'scenario': 'baseline', 'start': 40.0, 'end': 60.0, 'records': 28183}, {'scenario': 'baseline', 'start': 60.0, 'end': 80.0, 'records': 26679}, {'scenario': 'baseline', 'start': 80.0, 'end': 100.0, 'records': 23402}, {'scenario': 'baseline', 'start': 100.0, 'end': 120.0, 'records': 19188}, {'scenario': 'baseline', 'start': 120.0, 'end': 140.0, 'records': 15162}, {'scenario': 'baseline', 'start': 140.0, 'end': 160.0, 'records': 12126}, {'scenario': 'baseline', 'start': 160.0, 'end': 180.0, 'records': 9417}, {'scenario': 'baseline', 'start': 180.0, 'end': 200.0, 'records': 7305}, {'scenario': 'baseline', 'start': 200.0, 'end': 220.0, 'records': 5485}, {'scenario': 'baseline', 'start': 220.0, 'end': 240.0, 'records': 4039}, {'scenario': 'baseline', 'start': 240.0, 'end': 260.0, 'records': 3120}, {'scenario': 'baseline', 'start': 260.0, 'end': 280.0, 'records': 2333}, {'scenario': 'baseline', 'start': 280.0, 'end': 300.0, 'records': 1820}, {'scenario': 'baseline', 'start': 300.0, 'end': 320.0, 'records': 1340}, {'scenario': 'baseline', 'start': 320.0, 'end': 340.0, 'records': 1030}, {'scenario': 'baseline', 'start': 340.0, 'end': 360.0, 'records': 787}, {'scenario': 'baseline', 'start': 360.0, 'end': 380.0, 'records': 576}, {'scenario': 'baseline', 'start': 380.0, 'end': 400.0, 'records': 443}, {'scenario': 'baseline', 'start': 400.0, 'end': 420.0, 'records': 1463}, {'scenario': 'premium_0.02', 'start': 0.0, 'end': 20.0, 'records': 11646}, {'scenario': 'premium_0.02', 'start': 20.0, 'end': 40.0, 'records': 24942}, {'scenario': 'premium_0.02', 'start': 40.0, 'end': 60.0, 'records': 28372}, {'scenario': 'premium_0.02', 'start': 60.0, 'end': 80.0, 'records': 26869}, {'scenario': 'premium_0.02', 'start': 80.0, 'end': 100.0, 'records': 23525}, {'scenario': 'premium_0.02', 'start': 100.0, 'end': 120.0, 'records': 19160}, {'scenario': 'premium_0.02', 'start': 120.0, 'end': 140.0, 'records': 15098}, {'scenario': 'premium_0.02', 'start': 140.0, 'end': 160.0, 'records': 12087}, {'scenario': 'premium_0.02', 'start': 160.0, 'end': 180.0, 'records': 9318}, {'scenario': 'premium_0.02', 'start': 180.0, 'end': 200.0, 'records': 7223}, {'scenario': 'premium_0.02', 'start': 200.0, 'end': 220.0, 'records': 5348}, {'scenario': 'premium_0.02', 'start': 220.0, 'end': 240.0, 'records': 3978}, {'scenario': 'premium_0.02', 'start': 240.0, 'end': 260.0, 'records': 3010}, {'scenario': 'premium_0.02', 'start': 260.0, 'end': 280.0, 'records': 2312}, {'scenario': 'premium_0.02', 'start': 280.0, 'end': 300.0, 'records': 1738}, {'scenario': 'premium_0.02', 'start': 300.0, 'end': 320.0, 'records': 1291}, {'scenario': 'premium_0.02', 'start': 320.0, 'end': 340.0, 'records': 989}, {'scenario': 'premium_0.02', 'start': 340.0, 'end': 360.0, 'records': 745}, {'scenario': 'premium_0.02', 'start': 360.0, 'end': 380.0, 'records': 561}, {'scenario': 'premium_0.02', 'start': 380.0, 'end': 400.0, 'records': 430}, {'scenario': 'premium_0.02', 'start': 400.0, 'end': 420.0, 'records': 1358}, {'scenario': 'premium_0.06', 'start': 0.0, 'end': 20.0, 'records': 11365}, {'scenario': 'premium_0.06', 'start': 20.0, 'end': 40.0, 'records': 24311}, {'scenario': 'premium_0.06', 'start': 40.0, 'end': 60.0, 'records': 27909}, {'scenario': 'premium_0.06', 'start': 60.0, 'end': 80.0, 'records': 26569}, {'scenario': 'premium_0.06', 'start': 80.0, 'end': 100.0, 'records': 23321}, {'scenario': 'premium_0.06', 'start': 100.0, 'end': 120.0, 'records': 19143}, {'scenario': 'premium_0.06', 'start': 120.0, 'end': 140.0, 'records': 15211}, {'scenario': 'premium_0.06', 'start': 140.0, 'end': 160.0, 'records': 12128}, {'scenario': 'premium_0.06', 'start': 160.0, 'end': 180.0, 'records': 9541}, {'scenario': 'premium_0.06', 'start': 180.0, 

#### unsecured_rate

`histogram` of `rate` at steps matching `loans/pricing/unsecured`

scenario,column,edges,counts,below,above,nulls
str,str,str,str,i64,i64,i64
"""baseline""","""rate""","""[0.1, 0.1125, 0.125, 0.1375, 0…","""[0, 0, 0, 10838, 26335, 22676,…",0,0,0
"""premium_0.02""","""rate""","""[0.1, 0.1125, 0.125, 0.1375, 0…","""[0, 4949, 32224, 27513, 12220,…",0,0,0
"""premium_0.06""","""rate""","""[0.1, 0.1125, 0.125, 0.1375, 0…","""[0, 0, 0, 0, 587, 14629, …] (2…",0,0,0
"""premium_0.08""","""rate""","""[0.1, 0.1125, 0.125, 0.1375, 0…","""[0, 0, 0, 0, 0, 0, …] (20)""",0,0,0


Figure(grammar='vega-lite', spec={'config': {'view': {'continuousWidth': 300, 'continuousHeight': 300}}, 'data': {'values': [{'scenario': 'baseline', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'baseline', 'start': 0.1125, 'end': 0.125, 'records': 0}, {'scenario': 'baseline', 'start': 0.125, 'end': 0.1375, 'records': 0}, {'scenario': 'baseline', 'start': 0.1375, 'end': 0.15, 'records': 10838}, {'scenario': 'baseline', 'start': 0.15, 'end': 0.1625, 'records': 26335}, {'scenario': 'baseline', 'start': 0.1625, 'end': 0.175, 'records': 22676}, {'scenario': 'baseline', 'start': 0.175, 'end': 0.1875, 'records': 13279}, {'scenario': 'baseline', 'start': 0.1875, 'end': 0.2, 'records': 5427}, {'scenario': 'baseline', 'start': 0.2, 'end': 0.2125, 'records': 1309}, {'scenario': 'baseline', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 133}, {'scenario': 'baseline', 'start': 0.22499999999999998, 'end': 0.2375, 'records': 0}, {'scenario': 'baseline', 'start': 0.2375, 'end': 0.25, 'records': 0}, {'scenario': 'baseline', 'start': 0.25, 'end': 0.26249999999999996, 'records': 0}, {'scenario': 'baseline', 'start': 0.26249999999999996, 'end': 0.275, 'records': 0}, {'scenario': 'baseline', 'start': 0.275, 'end': 0.2875, 'records': 0}, {'scenario': 'baseline', 'start': 0.2875, 'end': 0.3, 'records': 0}, {'scenario': 'baseline', 'start': 0.3, 'end': 0.3125, 'records': 0}, {'scenario': 'baseline', 'start': 0.3125, 'end': 0.32499999999999996, 'records': 0}, {'scenario': 'baseline', 'start': 0.32499999999999996, 'end': 0.3375, 'records': 0}, {'scenario': 'baseline', 'start': 0.3375, 'end': 0.35, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.1125, 'end': 0.125, 'records': 4949}, {'scenario': 'premium_0.02', 'start': 0.125, 'end': 0.1375, 'records': 32224}, {'scenario': 'premium_0.02', 'start': 0.1375, 'end': 0.15, 'records': 27513}, {'scenario': 'premium_0.02', 'start': 0.15, 'end': 0.1625, 'records': 12220}, {'scenario': 'premium_0.02', 'start': 0.1625, 'end': 0.175, 'records': 2903}, {'scenario': 'premium_0.02', 'start': 0.175, 'end': 0.1875, 'records': 188}, {'scenario': 'premium_0.02', 'start': 0.1875, 'end': 0.2, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2, 'end': 0.2125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.22499999999999998, 'end': 0.2375, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2375, 'end': 0.25, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.25, 'end': 0.26249999999999996, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.26249999999999996, 'end': 0.275, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.275, 'end': 0.2875, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.2875, 'end': 0.3, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3, 'end': 0.3125, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3125, 'end': 0.32499999999999996, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.32499999999999996, 'end': 0.3375, 'records': 0}, {'scenario': 'premium_0.02', 'start': 0.3375, 'end': 0.35, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1, 'end': 0.1125, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1125, 'end': 0.125, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.125, 'end': 0.1375, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.1375, 'end': 0.15, 'records': 0}, {'scenario': 'premium_0.06', 'start': 0.15, 'end': 0.1625, 'records': 587}, {'scenario': 'premium_0.06', 'start': 0.1625, 'end': 0.175, 'records': 14629}, {'scenario': 'premium_0.06', 'start': 0.175, 'end': 0.1875, 'records': 21957}, {'scenario': 'premium_0.06', 'start': 0.1875, 'end': 0.2, 'records': 19133}, {'scenario': 'premium_0.06', 'start': 0.2, 'end': 0.2125, 'records': 13162}, {'scenario': 'premium_0.06', 'start': 0.2125, 'end': 0.22499999999999998, 'records': 6874}, 